# Best vs worst shot contests (SCQ extremes)

Same pipeline as `shot_viz_from_dataset.ipynb`: **three strongest** then **three weakest** contests by `shot_contest_quality` (higher = stronger contest).

For each extreme, the helper shows **three visuals in order**: (1) **static release image** (matplotlib half/full court), (2) **rotatable 3D at release** (Plotly), (3) **full sequence through make/miss** (Plotly slider: pre-release tail + frames up to outcome).

**OneDrive / cloud parquet:** timeouts often come from hitting the drive many times at once. This notebook passes a shared **`PARQUET_GAME_CACHE`** into `prepare_shot_viz_assets` so **each game file is read once** (and retried briefly on failure inside `shot_viz_from_dataset.py`).

Run cells **top to bottom** (or Restart & Run All). Each subsection is **its own cell** so you can pause between games if needed.

**Eligibility:** best/worst lists use only rows with **`analysis_eligible == yes`** after you rebuild the CSV with the updated pipeline (pass-like / half-court releases are flagged `no`).



In [10]:

import os
import subprocess
import sys
from pathlib import Path

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "nbformat>=5.10"], check=False)
try:
    import _plotly_utils.optional_imports as _poui
    _poui._not_importable.discard("nbformat")
except Exception:
    pass
import nbformat  # noqa: F401

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display


def _repo_root() -> Path:
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "scripts").is_dir() and (p / "data").is_dir():
            return p
    return here.parent


ROOT = _repo_root()
_viz_dir = ROOT / "scripts" / "visualization"
_viz_script = _viz_dir / "shot_viz_from_dataset.py"
if not _viz_script.is_file():
    raise FileNotFoundError(f"Expected {_viz_script}")
sys.path.insert(0, str(_viz_dir))

import importlib.util
for _m in ("shot_viz_from_dataset", "viz"):
    sys.modules.pop(_m, None)
_spec = importlib.util.spec_from_file_location("shot_viz_from_dataset", _viz_script)
if _spec is None or _spec.loader is None:
    raise ImportError(_viz_script)
_sv = importlib.util.module_from_spec(_spec)
sys.modules["shot_viz_from_dataset"] = _sv
_spec.loader.exec_module(_sv)

for _name in (
    "display_release_png_inline",
    "prepare_shot_viz_assets",
    "process_one_shot",
    "contest_quality_extreme_pick_list",
):
    if not hasattr(_sv, _name):
        raise ImportError(_name)

display_release_png_inline = _sv.display_release_png_inline
prepare_shot_viz_assets = _sv.prepare_shot_viz_assets
process_one_shot = _sv.process_one_shot
contest_quality_extreme_pick_list = _sv.contest_quality_extreme_pick_list

import plotly.io._renderers as _plotly_ir
_plotly_ir.nbformat = nbformat
import plotly.io as pio
pio.renderers.default = "vscode" if os.environ.get("VSCODE_PID") else "notebook_connected"



[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [11]:

# Match paths on your machine (copy from shot_viz_from_dataset.ipynb if needed)

SHOTS_CSV = ROOT / "data" / "outputs" / "datasets" / "shot_contest_dataset.csv"
PARQUET_DIR = "/Users/mariaangellobon/Library/CloudStorage/OneDrive-SharedLibraries-MassachusettsInstituteofTechnology/[MIT] Basketball Officiating - miami_heat_2025"
OUTPUT_DIR = ROOT / "data" / "outputs" / "visualizations" / "shot_sequences" / "runs_extremes"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PRE_FRAMES = 120
POST_FRAMES = 6
COURT_PNG_VIEW = "half"
COURT_PNG_HALF = None
N_EXTREMES = 3

# Resolved-path -> filtered game DataFrame (shared across all cells below)
PARQUET_GAME_CACHE: dict[str, pd.DataFrame] = {}


In [12]:
assert SHOTS_CSV.is_file(), f"Missing CSV: {SHOTS_CSV}"
shots = pd.read_csv(SHOTS_CSV, dtype={"game_id": str, "game_file": str})
EXTREME_PICKS = contest_quality_extreme_pick_list(shots, n=N_EXTREMES)
if not EXTREME_PICKS:
    raise RuntimeError("No rows with numeric shot_contest_quality.")
if len(EXTREME_PICKS) % 2 != 0:
    raise RuntimeError(f"Internal: expected paired picks, got {len(EXTREME_PICKS)}")
half = len(EXTREME_PICKS) // 2
BEST_PICKS = EXTREME_PICKS[:half]
WORST_PICKS = EXTREME_PICKS[half:]
if len(BEST_PICKS) < N_EXTREMES:
    raise RuntimeError(
        f"Only {len(BEST_PICKS)} ranked extremes per side (need N_EXTREMES={N_EXTREMES}). Lower N_EXTREMES or add more CSV rows."
    )
print("Strongest:")
for lab, _ in BEST_PICKS:
    print(" ", lab)
print("Weakest:")
for lab, _ in WORST_PICKS:
    print(" ", lab)


Strongest:
  Best #1 (SCQ=73.45)
  Best #2 (SCQ=71.26)
  Best #3 (SCQ=70.52)
Weakest:
  Worst #1 (SCQ=2.50)
  Worst #2 (SCQ=0.83)
  Worst #3 (SCQ=0.61)


### Helper — build & display one extreme
Used by each subsection cell so every shot reads parquet through **`PARQUET_GAME_CACHE`**.


In [13]:
def display_extreme_assets(pick: tuple) -> None:
    """pick = (label, shot_row Series) — release PNG, rotatable 3D at release, full trace through outcome."""
    label, shot_row = pick
    display(Markdown(f"### {label}"))
    assets = prepare_shot_viz_assets(
        shot_row,
        parquet_dir=str(PARQUET_DIR),
        pre_frames=PRE_FRAMES,
        post_frames=POST_FRAMES,
        court_png_view=COURT_PNG_VIEW,
        court_png_half=COURT_PNG_HALF,
        parquet_game_df_cache=PARQUET_GAME_CACHE,
    )
    display(Markdown("**1 — Release (half-/full-court PNG)**"))
    display_release_png_inline(assets["fig_matplotlib_release"])
    display(Markdown("**2 — Release (rotatable 3D, Plotly)**"))
    assets["fig_plotly_release"].show()
    display(Markdown("**3 — Through make/miss (Plotly; tail to outcome / PBP window)**"))
    assets["fig_plotly_window"].show()
    plt.close(assets["fig_matplotlib_release"])


## Strongest contests (highest SCQ)

Three separate cells so you need not rerun everything — run downward when ready.


### Strongest — #1


In [14]:
_idx = 0
if len(BEST_PICKS) > _idx:
    display_extreme_assets(BEST_PICKS[_idx])
else:
    display(Markdown('*No row — lower N_EXTREMES or check CSV.*'))


### Best #1 (SCQ=73.45)

TimeoutError: Timed out reading parquet after 5 attempt(s): '/Users/mariaangellobon/Library/CloudStorage/OneDrive-SharedLibraries-MassachusettsInstituteofTechnology/[MIT] Basketball Officiating - miami_heat_2025/nba_game_0022500901_processed.parquet'

Errno 60 is ETIMEDOUT: the filesystem did not return bytes in time while PyArrow opened or read the file. This is typical for **cloud-synced paths** (OneDrive, iCloud, Dropbox): the client may still be downloading, the file may only exist in the cloud, or the link stalled briefly.

**Fix:** put `*.parquet` in a **plain local folder** (fully on disk), set `PARQUET_DIR` to that path, or make the OneDrive folder **Available offline** / wait until sync finishes. The in-memory cache helps only *after* a game file has loaded once.

### Strongest — #2


In [ ]:
_idx = 1
if len(BEST_PICKS) > _idx:
    display_extreme_assets(BEST_PICKS[_idx])
else:
    display(Markdown('*No row — lower N_EXTREMES or check CSV.*'))


### Strongest — #3


In [ ]:
_idx = 2
if len(BEST_PICKS) > _idx:
    display_extreme_assets(BEST_PICKS[_idx])
else:
    display(Markdown('*No row — lower N_EXTREMES or check CSV.*'))


## Weakest contests (lowest SCQ)

Still uses the same cache — games loaded above are reused.


### Weakest — #1


In [ ]:
_idx = 0
if len(WORST_PICKS) > _idx:
    display_extreme_assets(WORST_PICKS[_idx])
else:
    display(Markdown('*No row — lower N_EXTREMES or check CSV.*'))


### Weakest — #2


In [ ]:
_idx = 1
if len(WORST_PICKS) > _idx:
    display_extreme_assets(WORST_PICKS[_idx])
else:
    display(Markdown('*No row — lower N_EXTREMES or check CSV.*'))


### Weakest — #3


In [ ]:
_idx = 2
if len(WORST_PICKS) > _idx:
    display_extreme_assets(WORST_PICKS[_idx])
else:
    display(Markdown('*No row — lower N_EXTREMES or check CSV.*'))


### Optional — save all six shots to disk
Leaves **`PARQUET_GAME_CACHE`** warm so these avoid extra reads when caches already filled.


In [ ]:

SAVE_TO_DISK = False  # flip to True once

if SAVE_TO_DISK:
    for label, shot_row in EXTREME_PICKS:
        display(Markdown(f"Saving: {label}"))
        process_one_shot(
            shot_row,
            parquet_dir=str(PARQUET_DIR),
            output_dir=str(OUTPUT_DIR),
            pre_frames=PRE_FRAMES,
            post_frames=POST_FRAMES,
            court_png_view=COURT_PNG_VIEW,
            court_png_half=COURT_PNG_HALF,
            parquet_game_df_cache=PARQUET_GAME_CACHE,
        )
    print("Wrote outputs under:", OUTPUT_DIR)
else:
    print("Set SAVE_TO_DISK = True to export PNG/HTML.")
